In [3]:
import pandas as pd
import numpy as np
import joblib
import gc
import traceback
from datetime import datetime
from darts import TimeSeries
from darts.models import RandomForestModel, RegressionModel, XGBModel, LightGBMModel
from darts.dataprocessing.transformers import Scaler, Diff
from darts.utils.missing_values import fill_missing_values
from sklearn.ensemble import ExtraTreesRegressor
import warnings

warnings.filterwarnings("ignore")

# Load data & tuning results
df_merged = joblib.load("saved_models/df_merged_20260326_1654.joblib")
TUNING_RESULTS = joblib.load("saved_models/optuna_tuning_results.joblib")

LEVEL_VARS = ["M2", "USDIDR", "Coal", "Copper", "Nickel", "Silver", "Tin", "STI", "Gold"]
RATE_VARS = ["BI_Rate", "CPI", "NPL_Ratio"]

print(f"Data: {df_merged.shape}")
for m, r in TUNING_RESULTS.items():
    print(f"  {m}: MAPE={r['best_value']:.4f}%")


Data: (2443, 14)
  RandomForest: MAPE=0.6336%
  ExtraTrees: MAPE=0.6329%
  XGBoost: MAPE=0.6329%
  LightGBM: MAPE=0.6329%


In [4]:
SINGLE_COVARIATES = {
    "None":      [],
    "BI_Rate":   ["BI_Rate"],
    "CPI":       ["CPI"],
    "M2":        ["M2"],
    "NPL_Ratio": ["NPL_Ratio"],
    "USDIDR":    ["USDIDR"],
    "Coal":      ["Coal"],
    "Copper":    ["Copper"],
    "Nickel":    ["Nickel"],
    "Silver":    ["Silver"],
    "Tin":       ["Tin"],
    "Gold":      ["Gold"],
    "STI":       ["STI"],
}

WINDOWS = [30, 120]
HORIZON = 1

total = len(SINGLE_COVARIATES) * len(WINDOWS) * 4
print(f"Total experiments: {total}")  # 13 × 2 × 4 = 104


Total experiments: 104


In [5]:
def to_series(df, target_col, covariates=None):
    target_series = TimeSeries.from_dataframe(
        df, time_col="date", value_cols=target_col,
        fill_missing_dates=True, freq="B"
    )
    target_series = fill_missing_values(target_series)
    covariate_series = None
    if covariates:
        covariate_series = TimeSeries.from_dataframe(
            df, time_col="date", value_cols=covariates,
            fill_missing_dates=True, freq="B"
        )
        covariate_series = fill_missing_values(covariate_series)
    return target_series, covariate_series


def build_model(model_name, params, window, has_covariates=True):
    common = {
        "lags": window,
        "lags_past_covariates": window if has_covariates else None,
        "output_chunk_length": HORIZON,
    }
    
    if model_name == "RandomForest":
        return RandomForestModel(**common, random_state=42, n_jobs=-1, **params)
    elif model_name == "ExtraTrees":
        return RegressionModel(
            lags=common["lags"],
            lags_past_covariates=common["lags_past_covariates"],
            output_chunk_length=common["output_chunk_length"],
            model=ExtraTreesRegressor(random_state=42, n_jobs=-1, **params)
        )
    elif model_name == "XGBoost":
        return XGBModel(**common, random_state=42, n_jobs=-1, **params)
    elif model_name == "LightGBM":
        return LightGBMModel(**common, random_state=42, n_jobs=-1, verbose=-1, **params)


def run_expanding_cv(model_name, params, target_series, covariate_series, window, n_folds=5):
    n = len(target_series)
    test_size = int(n * 0.15)
    min_train = int(n * 0.4)
    available = n - min_train - test_size
    step = max(1, available // (n_folds - 1))
    
    fold_metrics = {"mape": [], "mae": [], "rmse": [], "r2": [], "da": []}
    
    for fold in range(n_folds):
        train_end_idx = min_train + fold * step
        test_end_idx = min(train_end_idx + test_size, n)
        if test_end_idx > n:
            break
        
        train_ts = target_series[:train_end_idx]
        test_ts = target_series[train_end_idx:test_end_idx]
        fold_ts = target_series[:test_end_idx]
        
        # Log-diff target
        fold_log = fold_ts.map(np.log)
        train_log = train_ts.map(np.log)
        differencer = Diff(lags=1)
        train_log_diff = differencer.fit_transform(train_log)
        fold_log_diff = differencer.transform(fold_log)
        
        scaler = Scaler()
        train_scaled = scaler.fit_transform(train_log_diff)
        fold_scaled = scaler.transform(fold_log_diff)
        
        # Covariate transform
        cov_transformed = None
        if covariate_series is not None:
            cov_fold = covariate_series[:test_end_idx]
            cov_train = covariate_series[:train_end_idx]
            
            cov_cols = covariate_series.components.tolist()
            level_cols = [c for c in cov_cols if c in LEVEL_VARS]
            rate_cols = [c for c in cov_cols if c in RATE_VARS]
            
            parts_train, parts_fold = [], []
            
            if level_cols:
                d1 = Diff(lags=1)
                parts_train.append(d1.fit_transform(cov_train[level_cols].map(np.log)))
                parts_fold.append(d1.transform(cov_fold[level_cols].map(np.log)))
            
            if rate_cols:
                d2 = Diff(lags=1)
                parts_train.append(d2.fit_transform(cov_train[rate_cols]))
                parts_fold.append(d2.transform(cov_fold[rate_cols]))
            
            cov_train_combined = parts_train[0]
            cov_fold_combined = parts_fold[0]
            for p_tr, p_fo in zip(parts_train[1:], parts_fold[1:]):
                cov_train_combined = cov_train_combined.stack(p_tr)
                cov_fold_combined = cov_fold_combined.stack(p_fo)
            
            cov_scaler = Scaler()
            cov_scaler.fit(cov_train_combined)
            cov_transformed = cov_scaler.transform(cov_fold_combined)
        
        # Build & train
        has_cov = covariate_series is not None
        model = build_model(model_name, params, window, has_covariates=has_cov)
        model.fit(train_scaled, past_covariates=cov_transformed)
        
        # Predict
        forecast_list = model.historical_forecasts(
            series=fold_scaled, past_covariates=cov_transformed,
            start=test_ts.start_time(), forecast_horizon=1, stride=1,
            retrain=False, last_points_only=False, verbose=False
        )
        if isinstance(forecast_list, TimeSeries):
            forecast_list = [forecast_list]
        
        # Inverse transform
        all_pred_dates, all_pred_prices = [], []
        fold_log_full = fold_ts.map(np.log)
        for chunk in forecast_list:
            chunk_diff = scaler.inverse_transform(chunk)
            dates = chunk_diff.time_index
            vals = chunk_diff.values().flatten()
            idx = fold_ts.get_index_at_point(dates[0])
            anchor = fold_log_full[idx - 1].values()[0][0]
            prices = np.exp(anchor + np.cumsum(vals))
            all_pred_dates.extend(dates)
            all_pred_prices.extend(prices)
        
        # Metrics
        pred_df = pd.DataFrame({"date": pd.to_datetime(all_pred_dates), "predicted": all_pred_prices})
        actual_df = fold_ts.to_dataframe().reset_index()
        actual_df.columns = ["date", "actual"]
        eval_df = pd.merge(actual_df, pred_df, on="date", how="inner")
        
        y_true = eval_df["actual"].values
        y_pred = eval_df["predicted"].values
        
        fold_metrics["mape"].append(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
        fold_metrics["mae"].append(np.mean(np.abs(y_true - y_pred)))
        fold_metrics["rmse"].append(np.sqrt(np.mean((y_true - y_pred) ** 2)))
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - y_true.mean()) ** 2)
        fold_metrics["r2"].append(1 - ss_res / ss_tot)
        
        actual_dir = np.diff(y_true)
        pred_dir = y_pred[1:] - y_true[:-1]
        fold_metrics["da"].append(np.mean((actual_dir > 0) == (pred_dir > 0)) * 100)
        
        del model
        gc.collect()
    
    return fold_metrics

print("Functions ready.")


Functions ready.


In [6]:
results = []
start_time = datetime.now()
exp_num = 0
total_exp = len(SINGLE_COVARIATES) * len(WINDOWS) * len(TUNING_RESULTS)

print(f"Phase 1 Screening started: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total experiments: {total_exp}")
print("=" * 70)

for model_name, tune_res in TUNING_RESULTS.items():
    best_params = tune_res["best_params"]
    
    for cov_name, cov_vars in SINGLE_COVARIATES.items():
        for window in WINDOWS:
            exp_num += 1
            print(f"[{exp_num}/{total_exp}] {model_name} | {cov_name} | W{window}", end=" ")
            
            try:
                target_series, covariate_series = to_series(
                    df_merged, "IHSG", cov_vars if cov_vars else None
                )
                
                fold_metrics = run_expanding_cv(
                    model_name, best_params, target_series, covariate_series, window
                )
                
                results.append({
                    "Model": model_name,
                    "Covariates": cov_name,
                    "Window": window,
                    "MAPE_mean": round(np.mean(fold_metrics["mape"]), 4),
                    "MAPE_std": round(np.std(fold_metrics["mape"]), 4),
                    "MAE_mean": round(np.mean(fold_metrics["mae"]), 4),
                    "RMSE_mean": round(np.mean(fold_metrics["rmse"]), 4),
                    "RMSE_std": round(np.std(fold_metrics["rmse"]), 4),
                    "R2_mean": round(np.mean(fold_metrics["r2"]), 4),
                    "DA_mean": round(np.mean(fold_metrics["da"]), 2),
                    "DA_std": round(np.std(fold_metrics["da"]), 2),
                })
                
                print(f"→ MAPE={np.mean(fold_metrics['mape']):.4f}% | DA={np.mean(fold_metrics['da']):.1f}%")
                
            except Exception as e:
                traceback.print_exc()
                print(f"→ ERROR: {e}")
                results.append({
                    "Model": model_name, "Covariates": cov_name,
                    "Window": window, "Error": str(e)
                })

elapsed = datetime.now() - start_time
print(f"\nDone! Elapsed: {elapsed}")

df_results = pd.DataFrame(results)
df_results.to_csv("phase1_screening_results.csv", index=False)
display(df_results.sort_values("MAPE_mean"))


Phase 1 Screening started: 2026-03-27 10:24:18
Total experiments: 104
[1/104] RandomForest | None | W30 → MAPE=0.6604% | DA=50.3%
[2/104] RandomForest | None | W120 → MAPE=0.6587% | DA=51.0%
[3/104] RandomForest | BI_Rate | W30 → MAPE=0.6595% | DA=50.5%
[4/104] RandomForest | BI_Rate | W120 → MAPE=0.6583% | DA=52.1%
[5/104] RandomForest | CPI | W30 → MAPE=0.6591% | DA=50.2%
[6/104] RandomForest | CPI | W120 → MAPE=0.6584% | DA=51.9%
[7/104] RandomForest | M2 | W30 → MAPE=0.6592% | DA=50.4%
[8/104] RandomForest | M2 | W120 → MAPE=0.6583% | DA=51.0%
[9/104] RandomForest | NPL_Ratio | W30 → MAPE=0.6593% | DA=50.5%
[10/104] RandomForest | NPL_Ratio | W120 → MAPE=0.6583% | DA=51.9%
[11/104] RandomForest | USDIDR | W30 → MAPE=0.6587% | DA=52.2%
[12/104] RandomForest | USDIDR | W120 → MAPE=0.6587% | DA=50.6%
[13/104] RandomForest | Coal | W30 → MAPE=0.6593% | DA=51.3%
[14/104] RandomForest | Coal | W120 → MAPE=0.6594% | DA=50.7%
[15/104] RandomForest | Copper | W30 → MAPE=0.6586% | DA=51.1%
[

→ MAPE=0.6582% | DA=51.9%
[27/104] ExtraTrees | None | W30 

→ MAPE=0.6584% | DA=51.8%
[28/104] ExtraTrees | None | W120 

→ MAPE=0.6578% | DA=52.9%
[29/104] ExtraTrees | BI_Rate | W30 

→ MAPE=0.6580% | DA=52.3%
[30/104] ExtraTrees | BI_Rate | W120 

...

→ MAPE=0.6575% | DA=53.0%
[50/104] ExtraTrees | Gold | W120 

→ MAPE=0.6575% | DA=53.0%
[51/104] ExtraTrees | STI | W30 

→ MAPE=0.6579% | DA=52.5%
[52/104] ExtraTrees | STI | W120 

→ MAPE=0.6578% | DA=53.1%
[53/104] XGBoost | None | W30 → MAPE=0.6579% | DA=53.2%
[54/104] XGBoost | None | W120 → MAPE=0.6576% | DA=53.2%
[55/104] XGBoost | BI_Rate | W30 → MAPE=0.6579% | DA=53.2%
[56/104] XGBoost | BI_Rate | W120 → MAPE=0.6576% | DA=53.2%
[57/104] XGBoost | CPI | W30 → MAPE=0.6579% | DA=53.2%
[58/104] XGBoost | CPI | W120 → MAPE=0.6576% | DA=53.2%
[59/104] XGBoost | M2 | W30 → MAPE=0.6579% | DA=53.2%
[60/104] XGBoost | M2 | W120 → MAPE=0.6576% | DA=53.2%
[61/104] XGBoost | NPL_Ratio | W30 → MAPE=0.6579% | DA=53.2%
[62/104] XGBoost | NPL_Ratio | W120 → MAPE=0.6576% | DA=53.2%
[63/104] XGBoost | USDIDR | W30 → MAPE=0.6578% | DA=53.5%
[64/104] XGBoost | USDIDR | W120 → MAPE=0.6576% | DA=53.5%
[65/104] XGBoost | Coal | W30 → MAPE=0.6579% | DA=53.2%
[66/104] XGBoost | Coal | W120 → MAPE=0.6576% | DA=53.2%
[67/104] XGBoost | Copper | W30 → MAPE=0.6574% | DA=53.2%
[68/104] XGBoost | Copper | W120 → MAPE=0.6570% | DA=52.6%
[69/104] XGBoost | Nickel | W30 → MAPE=0.6580% | DA=

,Model,Covariates,Window,MAPE_mean,MAPE_std,MAE_mean,RMSE_mean,RMSE_std,R2_mean,DA_mean,DA_std
67,XGBoost,Copper,120,0.6570,0.1622,40.1960,54.9364,9.2643,0.9692,52.62,2.44
93,LightGBM,Copper,120,0.6571,0.1620,40.2036,54.9485,9.2559,0.9692,52.47,1.60
97,LightGBM,Silver,120,0.6572,0.1622,40.2129,54.9578,9.2930,0.9692,53.84,2.65
71,XGBoost,Silver,120,0.6572,0.1621,40.2112,54.9727,9.2786,0.9692,52.67,1.54
41,ExtraTrees,Copper,120,0.6573,0.1621,40.2169,54.9914,9.2834,0.9692,52.88,2.01
...,...,...,...,...,...,...,...,...,...,...,...
8,RandomForest,NPL_Ratio,30,0.6593,0.1632,40.3218,55.1017,9.3283,0.9690,50.48,1.20
12,RandomForest,Coal,30,0.6593,0.1629,40.3285,55.0558,9.3414,0.9691,51.35,1.91
13,RandomForest,Coal,120,0.6594,0.1620,40.3475,55.0685,9.2780,0.9690,50.69,0.63
2,RandomForest,BI_Rate,30,0.6595,0.1632,40.3318,55.1255,9.3432,0.9690,50.53,1.55


In [7]:
def analyze_screening(df_results, threshold_pct=0.3):
    screening = []
    
    for model_name in df_results["Model"].unique():
        for window in df_results["Window"].unique():
            mask = (df_results["Model"] == model_name) & (df_results["Window"] == window)
            subset = df_results[mask]
            
            baseline_mape = subset[subset["Covariates"] == "None"]["MAPE_mean"].values[0]
            
            for _, row in subset.iterrows():
                if row["Covariates"] == "None":
                    continue
                
                improvement = (baseline_mape - row["MAPE_mean"]) / baseline_mape * 100
                passed = improvement >= threshold_pct
                
                screening.append({
                    "Model": model_name,
                    "Window": window,
                    "Covariate": row["Covariates"],
                    "MAPE": row["MAPE_mean"],
                    "Baseline_MAPE": baseline_mape,
                    "Improvement_pct": round(improvement, 4),
                    "Passed": passed,
                })
    
    return pd.DataFrame(screening)

df_screening = analyze_screening(df_results, threshold_pct=0.3)

# Pass counts: berapa kali lolos dari 8 model×window combos
print("=== Pass Counts (out of 8 model×window combos) ===")
pass_counts = df_screening[df_screening["Passed"]].groupby("Covariate").size().sort_values(ascending=False)
print(pass_counts.to_string())

print("\n=== Average Improvement % ===")
avg_improvement = df_screening.groupby("Covariate")["Improvement_pct"].mean().sort_values(ascending=False)
print(avg_improvement.to_string())

display(df_screening.sort_values(["Covariate", "Model", "Window"]))


=== Pass Counts (out of 8 model×window combos) ===
Covariate
Gold      1
Silver    1

=== Average Improvement % ===
Covariate
Copper       0.113850
Silver       0.111900
Gold         0.083388
Tin          0.053062
STI          0.051150
CPI          0.043588
USDIDR       0.041663
M2           0.039800
BI_Rate      0.034125
NPL_Ratio    0.034112
Coal         0.020825
Nickel       0.009413


,Model,Window,Covariate,MAPE,Baseline_MAPE,Improvement_pct,Passed
24,ExtraTrees,30,BI_Rate,0.6580,0.6584,0.0608,False
36,ExtraTrees,120,BI_Rate,0.6577,0.6578,0.0152,False
72,LightGBM,30,BI_Rate,0.6579,0.6579,0.0000,False
84,LightGBM,120,BI_Rate,0.6576,0.6576,0.0000,False
0,RandomForest,30,BI_Rate,0.6595,0.6604,0.1363,False
...,...,...,...,...,...,...,...
88,LightGBM,120,USDIDR,0.6576,0.6576,0.0000,False
4,RandomForest,30,USDIDR,0.6587,0.6604,0.2574,False
16,RandomForest,120,USDIDR,0.6587,0.6587,0.0000,False
52,XGBoost,30,USDIDR,0.6578,0.6579,0.0152,False


In [8]:
# Covariates yang lolos screening
significant_macro = [v for v in ["BI_Rate", "CPI", "M2", "NPL_Ratio", "USDIDR"]
                     if v in pass_counts.index]
significant_commodity = [v for v in ["Coal", "Copper", "Nickel", "Silver", "Tin", "Gold"]
                         if v in pass_counts.index]
significant_regional = ["STI"] if "STI" in pass_counts.index else []
significant_all = significant_macro + significant_commodity + significant_regional

GROUP_COVARIATES = {}
if significant_macro:
    GROUP_COVARIATES["Sig_Macro"] = significant_macro
if significant_commodity:
    GROUP_COVARIATES["Sig_Commodity"] = significant_commodity
if significant_regional:
    GROUP_COVARIATES["Sig_Regional"] = significant_regional
if len(significant_all) > 1:
    GROUP_COVARIATES["Sig_All"] = significant_all

print("Phase 2 Groups:")
for k, v in GROUP_COVARIATES.items():
    print(f"  {k}: {v}")

# Save untuk Phase 2
joblib.dump(GROUP_COVARIATES, "saved_models/phase2_groups.joblib")
print("\nSaved: saved_models/phase2_groups.joblib")

df_screening.to_csv("phase1_screening_analysis.csv", index=False)
print("Saved: phase1_screening_analysis.csv")


Phase 2 Groups:
  Sig_Commodity: ['Silver', 'Gold']
  Sig_All: ['Silver', 'Gold']

Saved: saved_models/phase2_groups.joblib
Saved: phase1_screening_analysis.csv
